# 03 — Gold Layer Modeling
**Project**  : Olist Brazilian E-Commerce Data Warehouse  
**Layer**    : Gold (Star Schema)  
**Author**   : Salman  
**Platform** : Databricks (Unity Catalog + Delta Lake)  

---

## Purpose
This notebook builds the Gold layer using a **Star Schema** design.  
All tables are physical Delta Tables optimized for analytics and BI reporting.

## Star Schema Design
- **Grain** : 1 row = 1 item within 1 order
- **Fact**  : `fact_sales`
- **Dims**  : `dim_date`, `dim_customers`, `dim_products`, `dim_sellers`

```
                    dim_date
                   (date_key)
                       |
dim_customers      fact_sales      dim_products
(customer_key) --- (customer_key)  (product_key)
                   (product_key) --
                   (seller_key)  --
                   (date_key)       dim_sellers
                                   (seller_key)
```

## Surrogate Key Pattern
- All dimension tables use **surrogate keys** (integer, generated via `ROW_NUMBER()`) as primary keys
- Natural keys are **retained** in dimension tables for lookup during fact population
- Fact table stores **only surrogate keys** as foreign keys
- `order_id` and `order_item_id` are retained in fact as **degenerate dimensions**

## Build Order
Dimension tables must be built **before** fact table since fact_sales references surrogate keys from all dim tables.

| Step | Table | Depends On |
|------|-------|------------|
| 1 | dim_date | — |
| 2 | dim_customers | silver.customers, silver.geolocation |
| 3 | dim_products | silver.products, silver.product_category_name_translation |
| 4 | dim_sellers | silver.sellers, silver.geolocation |
| 5 | fact_sales | all dim tables + silver.order_items, orders, order_payments, order_reviews |

## Source → Target
- **Source** : `workspace.silver.*`
- **Target** : `workspace.gold.*`

## 0. Setup & Configuration

In [0]:
from datetime import datetime

def log(msg):
    print(f"[{datetime.now().strftime('%H:%M:%S')}] {msg}")

log("Setup complete.")
log("Source : workspace.silver")
log("Target : workspace.gold")

## 1. Create Gold Schema (if not exists)

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.gold")
log("Schema workspace.gold is ready.")

## 2. dim_date

**Source** : Generated from date range (no silver table)  
**Range**  : 2016-01-01 to 2018-12-31 (covers full Olist dataset period)  
**Note**   : Uses Spark native `sequence()` + `explode()` to generate date range — replaces recursive CTE from T-SQL version  
**Surrogate Key** : `date_key` — integer in `YYYYMMDD` format (e.g. 20180101)

In [0]:
start = datetime.now()
log(">> Building: dim_date")

spark.sql("""
    CREATE OR REPLACE TABLE workspace.gold.dim_date
    USING DELTA AS
    WITH date_range AS (
        SELECT explode(
            sequence(
                TO_DATE('2016-01-01'),
                TO_DATE('2018-12-31'),
                INTERVAL 1 DAY
            )
        ) AS full_date
    )
    SELECT
        CAST(DATE_FORMAT(full_date, 'yyyyMMdd') AS INT)  AS date_key,
        full_date,
        YEAR(full_date)                                  AS year,
        QUARTER(full_date)                               AS quarter,
        MONTH(full_date)                                 AS month,
        DATE_FORMAT(full_date, 'MMMM')                   AS month_name,
        DAY(full_date)                                   AS day,
        DAYOFWEEK(full_date)                             AS day_of_week,
        DATE_FORMAT(full_date, 'EEEE')                   AS day_name,
        CASE
            WHEN DAYOFWEEK(full_date) IN (1, 7) THEN 1
            ELSE 0
        END                                              AS is_weekend
    FROM date_range
""")

count = spark.sql("SELECT COUNT(*) AS cnt FROM workspace.gold.dim_date").collect()[0]["cnt"]
elapsed = (datetime.now() - start).seconds
log(f">> Done: gold.dim_date — {count:,} rows — {elapsed}s")

## 3. dim_customers

**Source** : `silver.customers` + `silver.geolocation`  
**Join**   : `customer_zip_code_prefix` → `geolocation_zip_code_prefix`  
**Note**   : City and state are taken from geolocation (cleaner), with fallback to customers if no match  
**Surrogate Key** : `customer_key` — generated via `ROW_NUMBER() OVER (ORDER BY customer_id)`

In [0]:
start = datetime.now()
log(">> Building: dim_customers")

spark.sql("""
    CREATE OR REPLACE TABLE workspace.gold.dim_customers
    USING DELTA AS
    SELECT
        ROW_NUMBER() OVER (ORDER BY c.customer_id)  AS customer_key,
        c.customer_id,
        c.customer_unique_id,
        c.customer_zip_code_prefix,
        COALESCE(g.geolocation_city,  c.customer_city)  AS customer_city,
        COALESCE(g.geolocation_state, c.customer_state) AS customer_state,
        g.geolocation_lat,
        g.geolocation_lng
    FROM workspace.silver.customers c
    LEFT JOIN workspace.silver.geolocation g
        ON c.customer_zip_code_prefix = g.geolocation_zip_code_prefix
""")

count = spark.sql("SELECT COUNT(*) AS cnt FROM workspace.gold.dim_customers").collect()[0]["cnt"]
elapsed = (datetime.now() - start).seconds
log(f">> Done: gold.dim_customers — {count:,} rows — {elapsed}s")

## 4. dim_products

**Source** : `silver.products` + `silver.product_category_name_translation`  
**Join**   : `product_category_name` → `product_category_name`  
**Note**   : English category name falls back to `'unknown'` if no translation match  
**Surrogate Key** : `product_key` — generated via `ROW_NUMBER() OVER (ORDER BY product_id)`

In [0]:
start = datetime.now()
log(">> Building: dim_products")

spark.sql("""
    CREATE OR REPLACE TABLE workspace.gold.dim_products
    USING DELTA AS
    SELECT
        ROW_NUMBER() OVER (ORDER BY p.product_id)       AS product_key,
        p.product_id,
        p.product_category_name,
        COALESCE(
            t.product_category_name_english, 'unknown'
        )                                               AS product_category_name_english,
        p.product_weight_g,
        p.product_length_cm,
        p.product_height_cm,
        p.product_width_cm
    FROM workspace.silver.products p
    LEFT JOIN workspace.silver.product_category_name_translation t
        ON p.product_category_name = t.product_category_name
""")

count = spark.sql("SELECT COUNT(*) AS cnt FROM workspace.gold.dim_products").collect()[0]["cnt"]
elapsed = (datetime.now() - start).seconds
log(f">> Done: gold.dim_products — {count:,} rows — {elapsed}s")

## 5. dim_sellers

**Source** : `silver.sellers` + `silver.geolocation`  
**Join**   : `seller_zip_code_prefix` → `geolocation_zip_code_prefix`  
**Note**   : City and state are taken from geolocation (cleaner), with fallback to sellers if no match.  
This resolves inconsistent `seller_city` format issues carried over from bronze.  
**Surrogate Key** : `seller_key` — generated via `ROW_NUMBER() OVER (ORDER BY seller_id)`

In [0]:
start = datetime.now()
log(">> Building: dim_sellers")

spark.sql("""
    CREATE OR REPLACE TABLE workspace.gold.dim_sellers
    USING DELTA AS
    SELECT
        ROW_NUMBER() OVER (ORDER BY s.seller_id)        AS seller_key,
        s.seller_id,
        s.seller_zip_code_prefix,
        COALESCE(g.geolocation_city,  s.seller_city)    AS seller_city,
        COALESCE(g.geolocation_state, s.seller_state)   AS seller_state,
        g.geolocation_lat,
        g.geolocation_lng
    FROM workspace.silver.sellers s
    LEFT JOIN workspace.silver.geolocation g
        ON s.seller_zip_code_prefix = g.geolocation_zip_code_prefix
""")

count = spark.sql("SELECT COUNT(*) AS cnt FROM workspace.gold.dim_sellers").collect()[0]["cnt"]
elapsed = (datetime.now() - start).seconds
log(f">> Done: gold.dim_sellers — {count:,} rows — {elapsed}s")

## 6. fact_sales

**Source** : `silver.order_items` (base) + `silver.orders` + `silver.order_payments` (pre-aggregated) + `silver.order_reviews`  
**Grain**  : 1 row = 1 item within 1 order (`order_id` + `order_item_id`)  

**Surrogate Key Lookup**:
- `customer_key` → JOIN `gold.dim_customers` ON `customer_id`
- `product_key`  → JOIN `gold.dim_products`  ON `product_id`
- `seller_key`   → JOIN `gold.dim_sellers`   ON `seller_id`
- `date_key`     → JOIN `gold.dim_date`      ON `CAST(DATE_FORMAT(order_purchase_timestamp, 'yyyyMMdd') AS INT)`

**Degenerate Dimensions**: `order_id`, `order_item_id` retained for traceability and BI filtering  

**Notes**:
- `order_payments` is pre-aggregated per `order_id` to avoid fanout (1 order can have multiple payment methods)
- `delivery_time_days` : days from purchase to delivered (NULL if not yet delivered)
- `delivery_delay_days`: positive = late, negative = early vs estimated delivery
- `is_late`            : 1 if delivered after estimated date, 0 if on-time, NULL if not delivered

In [0]:
start = datetime.now()
log(">> Building: fact_sales")

spark.sql("""
    CREATE OR REPLACE TABLE workspace.gold.fact_sales
    USING DELTA AS
    WITH payment_agg AS (
        -- Pre-aggregate payments per order to avoid fanout
        -- 1 order can have multiple payment methods (split payment)
        SELECT
            order_id,
            SUM(payment_value)                                              AS total_payment_value,
            COUNT(DISTINCT payment_sequential)                              AS payment_methods_count,
            MAX(CASE WHEN payment_type = 'credit_card' THEN 1 ELSE 0 END)  AS has_credit_card,
            MAX(CASE WHEN payment_type = 'voucher'     THEN 1 ELSE 0 END)  AS has_voucher,
            MAX(CASE WHEN payment_type = 'boleto'      THEN 1 ELSE 0 END)  AS has_boleto,
            MAX(CASE WHEN payment_type = 'debit_card'  THEN 1 ELSE 0 END)  AS has_debit_card
        FROM workspace.silver.order_payments
        GROUP BY order_id
    )
    SELECT
        -- Degenerate dimensions (transaction keys)
        oi.order_id,
        oi.order_item_id,

        -- Surrogate foreign keys (looked up from dim tables)
        dc.customer_key,
        dp.product_key,
        ds.seller_key,
        dd.date_key                                                         AS order_date_key,

        -- Order attributes
        o.order_status,
        o.order_purchase_timestamp,
        o.order_estimated_delivery_date,
        o.order_delivered_customer_date,

        -- Measures from order_items
        oi.price,
        oi.freight_value,

        -- Measures from order_payments (pre-aggregated)
        p.total_payment_value,
        p.payment_methods_count,
        p.has_credit_card,
        p.has_voucher,
        p.has_boleto,
        p.has_debit_card,

        -- Measures from order_reviews
        r.review_score,

        -- Calculated measures
        DATEDIFF(
            o.order_delivered_customer_date,
            o.order_purchase_timestamp
        )                                                                   AS delivery_time_days,

        DATEDIFF(
            o.order_delivered_customer_date,
            o.order_estimated_delivery_date
        )                                                                   AS delivery_delay_days,

        CASE
            WHEN o.order_delivered_customer_date > o.order_estimated_delivery_date THEN 1
            WHEN o.order_delivered_customer_date IS NULL                           THEN NULL
            ELSE 0
        END                                                                 AS is_late

    FROM workspace.silver.order_items oi

    -- Get order attributes
    INNER JOIN workspace.silver.orders o
        ON oi.order_id = o.order_id

    -- Surrogate key lookup: customer
    LEFT JOIN workspace.gold.dim_customers dc
        ON o.customer_id = dc.customer_id

    -- Surrogate key lookup: product
    LEFT JOIN workspace.gold.dim_products dp
        ON oi.product_id = dp.product_id

    -- Surrogate key lookup: seller
    LEFT JOIN workspace.gold.dim_sellers ds
        ON oi.seller_id = ds.seller_id

    -- Surrogate key lookup: date
    LEFT JOIN workspace.gold.dim_date dd
        ON CAST(DATE_FORMAT(o.order_purchase_timestamp, 'yyyyMMdd') AS INT) = dd.date_key

    -- Payment info (pre-aggregated to avoid fanout)
    LEFT JOIN payment_agg p
        ON oi.order_id = p.order_id

    -- Review info
    LEFT JOIN workspace.silver.order_reviews r
        ON oi.order_id = r.order_id
""")

count = spark.sql("SELECT COUNT(*) AS cnt FROM workspace.gold.fact_sales").collect()[0]["cnt"]
elapsed = (datetime.now() - start).seconds
log(f">> Done: gold.fact_sales — {count:,} rows — {elapsed}s")

## 7. Final Validation

In [0]:
log("=" * 65)
log("FINAL VALIDATION — GOLD LAYER")
log("=" * 65)

# ── Row counts ────────────────────────────────────────────────────────────────
tables = ["dim_date", "dim_customers", "dim_products", "dim_sellers", "fact_sales"]

print(f"\n{'Table':<20} {'Row Count':>12}")
print("-" * 35)
for tbl in tables:
    count = spark.sql(f"SELECT COUNT(*) AS cnt FROM workspace.gold.{tbl}").collect()[0]["cnt"]
    print(f"{tbl:<20} {count:>12,}")
print("-" * 35)

# ── Surrogate key uniqueness check ────────────────────────────────────────────
log("\nChecking surrogate key uniqueness...")

sk_checks = {
    "dim_date"      : "date_key",
    "dim_customers" : "customer_key",
    "dim_products"  : "product_key",
    "dim_sellers"   : "seller_key",
}

print(f"\n{'Table':<20} {'Key Column':<15} {'Duplicates':>12} {'Status':>8}")
print("-" * 58)
for tbl, key in sk_checks.items():
    dup = spark.sql(f"""
        SELECT COUNT(*) AS cnt
        FROM (
            SELECT {key}, COUNT(*) AS c
            FROM workspace.gold.{tbl}
            GROUP BY {key}
            HAVING COUNT(*) > 1
        )
    """).collect()[0]["cnt"]
    status = "[PASS]" if dup == 0 else "[FAIL]"
    print(f"{tbl:<20} {key:<15} {dup:>12,} {status:>8}")
print("-" * 58)

# ── Null surrogate key check in fact_sales ────────────────────────────────────
log("\nChecking NULL surrogate keys in fact_sales...")

fk_checks = ["customer_key", "product_key", "seller_key", "order_date_key"]

print(f"\n{'FK Column':<20} {'NULL Count':>12} {'Status':>8}")
print("-" * 43)
for fk in fk_checks:
    null_count = spark.sql(f"""
        SELECT COUNT(*) AS cnt
        FROM workspace.gold.fact_sales
        WHERE {fk} IS NULL
    """).collect()[0]["cnt"]
    status = "[PASS]" if null_count == 0 else "[WARN]"
    print(f"{fk:<20} {null_count:>12,} {status:>8}")
print("-" * 43)

log("\nValidation complete. Gold layer is ready for BI reporting.")